# Frozen-Embedding LOPO Harness (Medulloblastoma, GMCH Dataset)

Companion to `medulloblastoma_dsaf_net.ipynb`. Nothing here trains a network. Features come from
frozen extractors and a plain logistic regression sits on top, so a complete leave-one-patient-out
(LOPO) run takes seconds instead of an hour of retraining. That speed is what makes ablations
affordable with 11 patients.

**Questions this notebook is built to answer**
1. Does any frozen representation (ImageNet EfficientNetV2-S, Path Foundation, handcrafted texture)
   beat a trivial majority-class baseline under patient-level evaluation?
2. Does stain normalization / CLAHE help or hurt?
3. Is 10x, 100x, or both magnifications better?
4. Can simple colour statistics (6 numbers per image, no deep model) match the deep features? If yes,
   a model is probably reading stain or brightness, not morphology.
5. How much do image-level random splits inflate accuracy compared with patient-level LOPO on the
   same features and the same classifier?

**Scope, stated plainly.** After the 1766A/B merge there are 11 patients: Classic 5,
Desmoplastic_Nodular 3, Largecell 2, Normal 1. One patient is worth 10 percentage points of
patient-level accuracy for a 10-patient task. Treat the output as exploratory evidence, not as
a performance estimate. Normal has a single patient, so it can never be held out and still be
learned; the `all_classes` task reports it separately and excludes that fold from headline numbers.

Runs in a fresh kernel in the `medulloblastoma-dsaf` env. Your trained models in the other notebook's
kernel are not touched.

## 1. Setup and Configuration

In [1]:
import os, re, glob, json, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- paths -----------------------------------------------------------------
DATA_DIR = r"C:\Users\Shafique\Documents\uni\healthcare project\final_database"
OUTPUT_DIR = "./outputs"
FEATURE_CACHE_DIR = os.path.join(OUTPUT_DIR, "feature_cache")
os.makedirs(FEATURE_CACHE_DIR, exist_ok=True)
MAGNIFICATION_DIRS = ["10x", "100x"]
IMG_SIZE = (224, 224)

# ---- labels (same table as the main notebook; Desmoplastic + Nodular already merged) ----
SLIDE_LABELS = {
    "411": "Desmoplastic_Nodular", "203": "Desmoplastic_Nodular", "141": "Desmoplastic_Nodular",
    "321": "Classic", "279": "Classic", "277": "Classic",
    "1766A": "Classic", "1766B": "Classic", "188": "Classic",
    "17126": "Largecell", "395": "Largecell",
    "N": "Normal",
}
EXCLUDED_SELF_IDS = {"1609"}                              # no diagnosis given
PATIENT_GROUP_OVERRIDES = {"1766A": "1766", "1766B": "1766"}  # sub-slides of one case
PATIENT_ID_REGEX = r"^(.+?)\s*\(\d+\)"

# ---- experiment grid -----------------------------------------------------------------
TASKS = {
    "tumor_subtype": ["Classic", "Desmoplastic_Nodular", "Largecell"],             # 10 patients, all evaluable
    "all_classes":   ["Classic", "Desmoplastic_Nodular", "Largecell", "Normal"],   # Normal fold is unevaluable
}
MAGNIFICATIONS = {"10x": ["10x"], "100x": ["100x"], "both": ["10x", "100x"]}
PREPROC_CONFIGS = {                      # name -> (stain_norm, clahe)
    "raw": (False, False), "stain": (True, False), "clahe": (False, True), "stain+clahe": (True, True),
}
EXTRACTORS = ["efficientnetv2s", "pathfoundation", "texture", "color_stats"]
LOGREG_C = 0.1        # fairly strong L2; with ~350 tiles and hundreds of features, weak regularisation memorises
N_PERM = 100          # label-permutation runs for the significance check in section 6
PERM_CONFIG = dict(task="tumor_subtype", extractor="efficientnetv2s",
                   preproc="stain+clahe", magnification="both")   # fixed in advance, not picked from results

## 2. Manifest

In [2]:
def infer_patient_id(filename, regex):
    m = re.match(regex, filename)
    return m.group(1) if m else None

def build_manifest(data_dir, magnification_dirs, slide_labels, excluded_self_ids, patient_id_regex):
    rows, unmatched = [], set()
    for mag in magnification_dirs:
        mag_dir = os.path.join(data_dir, mag)
        if not os.path.isdir(mag_dir):
            print(f"[warn] magnification folder not found, skipping: {mag_dir}")
            continue
        for fp in sorted(glob.glob(os.path.join(mag_dir, "*.*"))):
            if not fp.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")):
                continue
            sid = infer_patient_id(os.path.basename(fp), patient_id_regex)
            if sid is None or sid in excluded_self_ids:
                continue
            label = slide_labels.get(sid)
            if label is None:
                unmatched.add(sid)
                continue
            rows.append({"filepath": fp, "label": label, "patient_id": sid, "magnification": mag})
    if unmatched:
        print(f"[warn] slide ids found on disk but not in SLIDE_LABELS (skipped): {sorted(unmatched)}")
    return pd.DataFrame(rows)

file_df = build_manifest(DATA_DIR, MAGNIFICATION_DIRS, SLIDE_LABELS, EXCLUDED_SELF_IDS, PATIENT_ID_REGEX)
assert len(file_df) > 0, "No images found - check DATA_DIR / MAGNIFICATION_DIRS / PATIENT_ID_REGEX"
file_df["split_group"] = file_df["patient_id"].map(lambda s: PATIENT_GROUP_OVERRIDES.get(s, s))

print(f"{len(file_df)} images, {file_df['split_group'].nunique()} patients")
print(file_df.groupby(["label", "magnification"]).size().unstack(fill_value=0))
print(file_df.groupby("label")["split_group"].nunique().rename("n_patients"))

357 images, 11 patients
magnification         100x  10x
label                          
Classic                 82   59
Desmoplastic_Nodular    22   65
Largecell               24   30
Normal                  25   50
label
Classic                 5
Desmoplastic_Nodular    3
Largecell               2
Normal                  1
Name: n_patients, dtype: int64


## 3. Preprocessing and Feature Extractors

Preprocessing functions are copied unchanged from the main notebook, except that stain/CLAHE are
explicit arguments so they can be ablated. Features are cached per (extractor, preprocessing) in
`outputs/feature_cache/` as `.npz` files keyed by file path, so re-running the grid costs nothing
after the first pass.

`color_stats` is a deliberate probe: LAB mean and std of the **raw** resized image (6 numbers).
If it scores near the deep features, those features are plausibly picking up stain/brightness.
It ignores the preprocessing grid, because normalising colour would erase the very signal it measures.

In [3]:
_REINHARD_REF = {"mean": np.array([65.0, 15.0, -5.0]), "std": np.array([12.0, 6.0, 6.0])}

def reinhard_stain_normalize(img_rgb, ref=_REINHARD_REF):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    for c in range(3):
        ch = lab[:, :, c]
        lab[:, :, c] = (ch - ch.mean()) * (ref["std"][c] / (ch.std() + 1e-6)) + ref["mean"][c]
    return cv2.cvtColor(np.clip(lab, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)

def apply_clahe(img_rgb, clip_limit=2.0, tile_grid_size=(8, 8)):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)

def preprocess_img(img_rgb, stain_norm, clahe, size=IMG_SIZE):
    img = cv2.resize(img_rgb, size, interpolation=cv2.INTER_AREA)
    if stain_norm:
        img = reinhard_stain_normalize(img)
    if clahe:
        img = apply_clahe(img)
    return img

def extract_texture_features(img_rgb, distances=(1,), angles=(0, np.pi/4, np.pi/2, 3*np.pi/4)):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    gray_q = (gray / 4).astype(np.uint8)
    glcm = graycomatrix(gray_q, distances=distances, angles=angles, levels=64, symmetric=True, normed=True)
    props = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]
    glcm_feats = np.concatenate([graycoprops(glcm, p).flatten() for p in props])
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    lbp_hist, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
    return np.concatenate([glcm_feats, lbp_hist]).astype(np.float32)

def color_stats_features(img_rgb):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(np.float32)
    return np.concatenate([lab.mean(axis=0), lab.std(axis=0)]).astype(np.float32)

In [4]:
import tensorflow as tf
from tensorflow import keras

_effnet = None
def _effnet_batch(imgs_uint8):
    global _effnet
    if _effnet is None:
        # ImageNet weights, frozen, global-average-pooled. EfficientNetV2 rescales internally,
        # so it takes raw 0-255 floats (same convention as the main notebook).
        _effnet = keras.applications.EfficientNetV2S(include_top=False, weights="imagenet",
                                                     pooling="avg", input_shape=(*IMG_SIZE, 3))
    return _effnet.predict(imgs_uint8.astype(np.float32), verbose=0)

_pf_infer = None
def _get_pf_infer():
    # Path Foundation, loaded lazily so cached features never trigger a download. Needs the HF
    # login done earlier. snapshot_download avoids huggingface_hub's keras_mixin (needs torch).
    global _pf_infer
    if _pf_infer is None:
        from huggingface_hub import snapshot_download
        d = snapshot_download(repo_id="google/path-foundation")
        pb = os.path.join(d, "saved_model.pb")
        size = os.path.getsize(pb) if os.path.exists(pb) else -1
        print(f"Path Foundation saved_model.pb: {size:,} bytes")
        if size < 1024:
            d = snapshot_download(repo_id="google/path-foundation", force_download=True)
        _pf_infer = tf.saved_model.load(d).signatures["serving_default"]
    return _pf_infer

def _pf_batch(imgs_uint8):
    infer = _get_pf_infer()
    out = []
    for img in imgs_uint8:
        t = tf.constant(img[None].astype(np.float32) / 255.0)   # Path Foundation expects [0, 1]
        res = infer(t)
        out.append((res["output_0"] if "output_0" in res else next(iter(res.values()))).numpy().ravel())
    return np.stack(out)

def _texture_batch(imgs_uint8):
    return np.stack([extract_texture_features(i) for i in imgs_uint8])

def _color_batch(imgs_uint8):
    return np.stack([color_stats_features(i) for i in imgs_uint8])

EXTRACTOR_SPECS = {
    "efficientnetv2s": dict(batch_fn=_effnet_batch, raw_only=False),
    "pathfoundation":  dict(batch_fn=_pf_batch,     raw_only=False),
    "texture":         dict(batch_fn=_texture_batch, raw_only=False),
    "color_stats":     dict(batch_fn=_color_batch,   raw_only=True),   # always raw, see section 3 note
}

def compute_features(paths, extractor, stain_norm, clahe, batch_size=32):
    fn = EXTRACTOR_SPECS[extractor]["batch_fn"]
    feats = []
    for i in range(0, len(paths), batch_size):
        imgs = []
        for p in paths[i:i + batch_size]:
            bgr = cv2.imread(p, cv2.IMREAD_COLOR)
            if bgr is None:
                raise FileNotFoundError(f"cv2 could not read {p}")
            imgs.append(preprocess_img(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), stain_norm, clahe))
        feats.append(fn(np.stack(imgs)))
    return np.concatenate(feats).astype(np.float32)

def get_features(extractor, preproc_name, df):
    # Returns an array aligned to df's row order. The cache is keyed by file path, so the glob
    # order of the manifest can change between runs without misaligning features.
    pp = "raw" if EXTRACTOR_SPECS[extractor]["raw_only"] else preproc_name
    cache = os.path.join(FEATURE_CACHE_DIR, f"{extractor}__{pp}.npz")
    paths = df["filepath"].tolist()
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=False)
        lookup = {p: i for i, p in enumerate(z["paths"].tolist())}
        if all(p in lookup for p in paths):
            return z["feats"][[lookup[p] for p in paths]]
    stain, clahe = PREPROC_CONFIGS[pp]
    feats = compute_features(paths, extractor, stain, clahe)
    np.savez(cache, paths=np.array(paths), feats=feats)
    return feats

## 4. Evaluation Harness

- **Classifier:** `StandardScaler` + L2 logistic regression with balanced class weights, inside one
  scikit-learn pipeline, so the scaler is fit on training tiles only in every fold.
- **LOPO:** one fold per patient (`split_group`). No inner validation split, so no fold can lose a
  class to it (the flaw in the main notebook's version).
- **Patient-level call:** majority vote over the held-out patient's tile predictions.
- **Baseline:** in each fold, predict the diagnosis shared by the most *training patients*
  (always Classic here). Reported with the same metrics, so the comparison is like for like.
  (A tile-count majority is avoided on purpose: under LOPO it is biased below chance.)
- **Metrics:** patient-level accuracy and patient-level *balanced* accuracy (mean of per-class
  recall over patients). Balanced accuracy matters here because Classic alone is 5 of 10 tumor patients.
- **Evaluable folds only:** a fold counts only if the held-out patient's class appears in the
  training data. This excludes the Normal fold in `all_classes`.
- **Image-level comparison:** the same features and classifier under stratified 5-fold CV over
  tiles, which lets tiles from one slide sit on both sides of the split.

In [5]:
# Per-tile predictions of the most recent lopo_evaluate / image_level_cv call, read by the grid loop
# (used to bootstrap CIs for the leakage gap; see scripts/gap_ci.py).
_LAST = {}

def _make_clf():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=LOGREG_C, class_weight="balanced", max_iter=2000))

def _patient_bal_acc(true, pred):
    # mean over classes of (fraction of that class's patients called correctly)
    per_class = [np.mean(pred[true == c] == c) for c in np.unique(true)]
    return float(np.mean(per_class)) if per_class else np.nan

def _patient_mode(y_tr, groups_tr, n_classes):
    # Baseline class = the diagnosis shared by the most TRAINING PATIENTS. A tile-count mode is
    # a trap under LOPO: holding a patient out makes that patient's class rarer in training, so
    # the baseline picks the wrong class and scores below chance, flattering any model.
    per_patient = pd.Series(y_tr).groupby(groups_tr).first().values
    return int(np.bincount(per_patient, minlength=n_classes).argmax())

def lopo_evaluate(X, y, groups, n_classes):
    logo = LeaveOneGroupOut()
    rows, pooled_true, pooled_pred = [], [], []
    _LAST["lopo"] = np.full(len(y), -1)      # -1 = tile in a non-evaluable fold
    for tr, te in logo.split(X, y, groups):
        held = int(y[te][0])
        evaluable = held in set(y[tr].tolist())
        pred = _make_clf().fit(X[tr], y[tr]).predict(X[te])
        rows.append(dict(patient=groups[te][0], true=held,
                         vote=int(np.bincount(pred, minlength=n_classes).argmax()),
                         base=_patient_mode(y[tr], groups[tr], n_classes),
                         n_tiles=len(te), tile_acc=float(np.mean(pred == y[te])),
                         evaluable=evaluable))
        if evaluable:
            pooled_true.append(y[te]); pooled_pred.append(pred)
            _LAST["lopo"][te] = pred
    folds = pd.DataFrame(rows)
    ev = folds[folds.evaluable]
    if len(ev) == 0:
        return dict(n_evaluable=0), folds
    t, v, b = ev["true"].values, ev["vote"].values, ev["base"].values
    pt, pp = np.concatenate(pooled_true), np.concatenate(pooled_pred)
    metrics = dict(
        n_evaluable=len(ev),
        patient_acc=float(np.mean(v == t)), patient_bal_acc=_patient_bal_acc(t, v),
        base_patient_acc=float(np.mean(b == t)), base_patient_bal_acc=_patient_bal_acc(t, b),
        tile_acc=float(accuracy_score(pt, pp)), tile_bal_acc=float(balanced_accuracy_score(pt, pp)),
    )
    return metrics, folds

def image_level_cv(X, y, n_splits=5):
    # Deliberately leaky: tiles from the same slide land in both train and test folds.
    n_splits = max(2, min(n_splits, int(np.bincount(y).min())))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    pred = np.empty_like(y)
    for tr, te in skf.split(X, y):
        pred[te] = _make_clf().fit(X[tr], y[tr]).predict(X[te])
    _LAST["img"] = pred.copy()
    return float(accuracy_score(y, pred)), float(balanced_accuracy_score(y, pred))

def build_task_arrays(df, X_all, task, magnification):
    classes = TASKS[task]
    mask = df["label"].isin(classes).values & df["magnification"].isin(MAGNIFICATIONS[magnification]).values
    sub = df[mask]
    y = sub["label"].map({c: i for i, c in enumerate(classes)}).values.astype(int)
    return X_all[mask], y, sub["split_group"].values, classes

def permutation_test(X, y, groups, n_classes, observed, n_perm=N_PERM):
    # Shuffle class labels across PATIENTS (all tiles of a patient move together), keeping class
    # counts, then rerun the full LOPO. Answers: how often does a label-free assignment score
    # at least as well as the real labels?
    rng = np.random.default_rng(SEED)
    patients = np.unique(groups)
    pat_label = np.array([y[groups == p][0] for p in patients])
    null = []
    for _ in range(n_perm):
        mapping = dict(zip(patients, rng.permutation(pat_label)))
        y_perm = np.array([mapping[g] for g in groups])
        m, _ = lopo_evaluate(X, y_perm, groups, n_classes)
        if m.get("n_evaluable", 0) > 0:
            null.append(m["patient_bal_acc"])
    null = np.array(null)
    p = (1 + np.sum(null >= observed - 1e-12)) / (len(null) + 1)
    return null, float(p)

## 5. Run the Grid

In [6]:
results, skipped = [], {}
TILE_LOG = []     # one block of per-tile predictions per tumor_subtype configuration
for task in TASKS:
    for ext in EXTRACTORS:
        pps = ["raw"] if EXTRACTOR_SPECS[ext]["raw_only"] else list(PREPROC_CONFIGS)
        for pp in pps:
            try:
                X_all = get_features(ext, pp, file_df)
            except Exception as e:
                if ext not in skipped:
                    skipped[ext] = f"{type(e).__name__}: {str(e)[:200]}"
                    print(f"[skip] extractor '{ext}' unavailable -> {skipped[ext]}")
                break
            for mag in MAGNIFICATIONS:
                X, y, g, classes = build_task_arrays(file_df, X_all, task, mag)
                if len(np.unique(g)) < 3:
                    continue
                m, _ = lopo_evaluate(X, y, g, len(classes))
                row = dict(task=task, extractor=ext, preproc=pp, magnification=mag,
                           n_patients=len(np.unique(g)), n_features=X.shape[1], **m)
                if task == "tumor_subtype":
                    row["imglevel_tile_acc"], row["imglevel_tile_bal_acc"] = image_level_cv(X, y)
                    TILE_LOG.append(pd.DataFrame(dict(extractor=ext, preproc=pp, magnification=mag,
                                                      patient_id=g, y=y, lopo_pred=_LAST["lopo"], img_pred=_LAST["img"])))
                results.append(row)
        print(f"done: {task} / {ext}")

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUTPUT_DIR, "frozen_embedding_lopo_results.csv"), index=False)
pd.concat(TILE_LOG, ignore_index=True).to_csv(os.path.join(OUTPUT_DIR, "frozen_tile_predictions.csv"), index=False)
print(f"\n{len(results_df)} configurations evaluated; skipped extractors: {list(skipped) or 'none'}")

done: tumor_subtype / efficientnetv2s
done: tumor_subtype / pathfoundation
done: tumor_subtype / texture
done: tumor_subtype / color_stats
done: all_classes / efficientnetv2s
done: all_classes / pathfoundation
done: all_classes / texture
done: all_classes / color_stats

78 configurations evaluated; skipped extractors: none


In [7]:
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
cols = ["extractor", "preproc", "magnification", "n_evaluable", "patient_acc", "patient_bal_acc",
        "base_patient_acc", "base_patient_bal_acc", "tile_bal_acc"]

tumor = results_df[results_df.task == "tumor_subtype"].copy()
print("TUMOR SUBTYPE (10 patients, 1 patient = 10 points) - top 12 by patient-level balanced accuracy")
print(tumor.sort_values("patient_bal_acc", ascending=False)[cols].head(12).round(3).to_string(index=False))

print("\nPreprocessing ablation (patient-level balanced accuracy):")
print(tumor.pivot_table(index=["extractor", "magnification"], columns="preproc",
                        values="patient_bal_acc").round(3))

print("\nMagnification (mean over preprocessing configs):")
print(tumor.pivot_table(index="extractor", columns="magnification", values="patient_bal_acc").round(3))

print("\nBaseline for reference (diagnosis shared by most training patients): patient bal. acc.",
      tumor["base_patient_bal_acc"].round(3).unique())

TUMOR SUBTYPE (10 patients, 1 patient = 10 points) - top 12 by patient-level balanced accuracy
      extractor     preproc magnification  n_evaluable  patient_acc  patient_bal_acc  base_patient_acc  base_patient_bal_acc  tile_bal_acc
 pathfoundation       stain          100x           10          0.5            0.433               0.5                 0.333         0.243
        texture       clahe          both           10          0.5            0.422               0.5                 0.333         0.342
        texture       stain          both           10          0.5            0.422               0.5                 0.333         0.310
    color_stats         raw          100x           10          0.5            0.422               0.5                 0.333         0.341
        texture stain+clahe          both           10          0.5            0.422               0.5                 0.333         0.318
efficientnetv2s       stain          both           10          0.5    

## 6. Leakage Gap and Significance

**Leakage gap.** Same features, same classifier, same tumor-subtype task. The only difference is
whether tiles from one slide may appear on both sides of the split. The gap is
`image-level - patient-level` balanced tile accuracy.

**Permutation check.** For the single configuration fixed in advance (`PERM_CONFIG`), class labels
are shuffled across patients `N_PERM` times and the full LOPO is rerun. The p-value is the share of
shuffles scoring at least as well as the real labels. It applies to that one configuration only. It does
**not** cover picking the best row out of the grid above, which is optimistic by construction
(many configurations, 10 patients).

In [8]:
leak = tumor.dropna(subset=["imglevel_tile_bal_acc"]).copy()
leak["leakage_gap"] = leak["imglevel_tile_bal_acc"] - leak["tile_bal_acc"]
print("Balanced TILE accuracy: patient-level LOPO vs leaky image-level CV (tumor subtype)")
print(leak.groupby("extractor")[["tile_bal_acc", "imglevel_tile_bal_acc", "leakage_gap"]]
          .mean().round(3).sort_values("leakage_gap", ascending=False))
print("\nWorst case per extractor:")
print(leak.loc[leak.groupby("extractor")["leakage_gap"].idxmax(),
               ["extractor", "preproc", "magnification", "tile_bal_acc",
                "imglevel_tile_bal_acc", "leakage_gap"]].round(3).to_string(index=False))

Balanced TILE accuracy: patient-level LOPO vs leaky image-level CV (tumor subtype)
                 tile_bal_acc  imglevel_tile_bal_acc  leakage_gap
extractor                                                        
pathfoundation          0.189                  0.891        0.702
efficientnetv2s         0.258                  0.797        0.539
texture                 0.175                  0.641        0.466
color_stats             0.214                  0.599        0.386

Worst case per extractor:
      extractor preproc magnification  tile_bal_acc  imglevel_tile_bal_acc  leakage_gap
    color_stats     raw           10x         0.105                  0.587        0.483
efficientnetv2s     raw           10x         0.211                  0.885        0.674
 pathfoundation   clahe           10x         0.123                  0.972        0.849
        texture     raw           10x         0.077                  0.752        0.675


In [9]:
pc = PERM_CONFIG
try:
    X_all = get_features(pc["extractor"], pc["preproc"], file_df)
    X, y, g, classes = build_task_arrays(file_df, X_all, pc["task"], pc["magnification"])
    obs, folds = lopo_evaluate(X, y, g, len(classes))
    null, p = permutation_test(X, y, g, len(classes), obs["patient_bal_acc"])
    print(f"Pre-specified config: {pc}")
    print(f"Observed patient-level balanced accuracy: {obs['patient_bal_acc']:.3f} "
          f"(baseline {obs['base_patient_bal_acc']:.3f})")
    print(f"Label-permutation null: mean {null.mean():.3f}, 95th pct {np.percentile(null, 95):.3f} "
          f"({len(null)} runs)")
    print(f"p = {p:.3f}")
    print("\nPer-patient calls for this configuration:")
    names = {i: c for i, c in enumerate(classes)}
    show = folds.assign(true=folds.true.map(names), vote=folds.vote.map(names), base=folds.base.map(names))
    print(show[["patient", "true", "vote", "base", "n_tiles", "tile_acc"]].round(3).to_string(index=False))
except Exception as e:
    print(f"[skip] permutation check not run: {type(e).__name__}: {e}")

Pre-specified config: {'task': 'tumor_subtype', 'extractor': 'efficientnetv2s', 'preproc': 'stain+clahe', 'magnification': 'both'}
Observed patient-level balanced accuracy: 0.333 (baseline 0.333)
Label-permutation null: mean 0.246, 95th pct 0.444 (100 runs)
p = 0.248

Per-patient calls for this configuration:
patient                 true                 vote    base  n_tiles  tile_acc
    141 Desmoplastic_Nodular              Classic Classic       25     0.000
  17126            Largecell Desmoplastic_Nodular Classic       10     0.100
   1766              Classic              Classic Classic       20     0.750
    188              Classic              Classic Classic       17     0.647
    203 Desmoplastic_Nodular              Classic Classic       24     0.458
    277              Classic              Classic Classic       11     0.455
    279              Classic              Classic Classic       37     0.595
    321              Classic              Classic Classic       56     0.

## 7. How to Read This

- With 10 tumor patients, a one-patient change moves patient accuracy by 10 points and balanced
  accuracy by more (Largecell has 2 patients, so one patient is 50% of that class). Differences smaller
  than that are noise. Look for patterns that repeat across extractors and magnifications, not for one top row.
- **Compare against the permutation null, not against 1/3.** Under LOPO the null sits *below* nominal
  chance, because leaving a patient out makes that patient's class rarer in training. Scoring 35%
  on a 3-class task can therefore still be better than chance, and 40% can still be unremarkable.
- The grid has up to ~100 configurations. The best row is selected from many, so it overstates
  real performance. The permutation check on the pre-specified configuration is the cleaner number.
- **If `color_stats` is competitive with the deep features,** the deep features are plausibly reading
  stain or brightness. That would also explain near-chance results on unseen patients.
- **If nothing beats the baseline,** that is the finding: with this cohort, patient-level subtype
  signal is not detectable with frozen features. The leakage gap in section 6 then shows what image-level
  splits would have claimed instead.
- **If Path Foundation is missing** (`[skip]` line in section 5), its model may have been exported with a
  newer TensorFlow than 2.9.1. You can compute its features in a newer environment and save them as
  `outputs/feature_cache/pathfoundation__<preproc>.npz` with arrays `paths` and `feats`; `get_features`
  will pick them up without loading the model.
- None of this validates a diagnostic tool. Two facts would change what is possible, and both need GMCH:
  whether the `N` images are one normal case or several, and whether 1766A/B are one patient.